# 05 - Random Forest Regressor

> Ensemble bagging model — aggregates many decision trees to reduce variance and improve generalization on the same lag/rolling features used by XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
COLORS = {'primary':'#003B73','secondary':'#0074B7','alert':'#BF212F','success':'#27AE60'}
FEATURES = ['month','quarter','year','trend','lag_1','lag_2','lag_3','lag_12','rolling_mean_3','rolling_mean_6','rolling_std_3']
print("Libraries loaded")

## 1. Load Prepared ML Data

In [ ]:
train = pd.read_csv('../data/train_ml.csv', parse_dates=['ds'])
test = pd.read_csv('../data/test_ml.csv', parse_dates=['ds'])
X_train, y_train = train[FEATURES], train['y']
X_test, y_test = test[FEATURES], test['y']
print(f"Train: {len(train)} months | Test: {len(test)} months")

## 2. Train Random Forest

In [ ]:
model = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_split=3,
                              min_samples_leaf=2, random_state=42)
model.fit(X_train, y_train)
print("Random Forest model trained")

## 3. Evaluate

In [ ]:
y_true = y_test.values
y_pred = model.predict(X_test)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
print(f"Random Forest Test Metrics:")
print(f"  MAPE:  {mape:.2f}%")
print(f"  MAE:   ${mae:,.2f}")
print(f"  RMSE:  ${rmse:,.2f}")
print(f"  R2:    {r2:.4f}")

## 4. Feature Importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(10,6))
imp.plot.barh(color=COLORS['success'], edgecolor='white')
plt.title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../visualizations/rf_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Visualize Predictions

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(test['ds'], y_true, label='Actual (Test)', color='black', linewidth=2, marker='o')
plt.plot(test['ds'], y_pred, label=f'RF Pred (MAPE={mape:.1f}%)', color=COLORS['success'], linestyle='--', linewidth=2, marker='^')
plt.title('Random Forest Forecast vs Actuals', fontsize=15, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Monthly Sales ($)'); plt.legend()
plt.tight_layout()
plt.savefig('../visualizations/rf_forecast.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Save Model & Predictions

In [ ]:
pred_df = pd.DataFrame({'ds': test['ds'].values, 'actual': y_true, 'predicted': y_pred})
pred_df.to_csv('../predictions/random_forest_predictions.csv', index=False)
joblib.dump(model, '../models/random_forest_model.pkl')
print("Saved: predictions/random_forest_predictions.csv")
print("Saved: models/random_forest_model.pkl")
print("\nProceed to 06_Model_Comparison.ipynb")